# 🐳 Docker Containerization for ML Models

> **Containerize machine learning applications for consistent deployment**

This notebook demonstrates how to containerize ML models using Docker, ensuring consistent deployment across different environments.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Create** Dockerfiles for ML applications
- **Build** and run Docker containers
- **Optimize** container size and performance
- **Handle** model artifacts and dependencies
- **Deploy** containers to production environments

In [ ]:
import os
import subprocess
import json
import requests
import time
from pathlib import Path

print("✅ All imports successful!")
print("📝 This notebook will create Docker configuration files")
print("🐳 Make sure Docker is installed and running on your system")

## 🐳 Creating Dockerfile for ML Application

In [ ]:
# Create Dockerfile for the ML Flask application
dockerfile_content = '''
# Use official Python runtime as base image
FROM python:3.9-slim

# Set working directory in container
WORKDIR /app

# Install system dependencies
RUN apt-get update && apt-get install -y \
    gcc \
    g++ \
    && rm -rf /var/lib/apt/lists/*

# Copy requirements first for better caching
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py .
COPY model.pkl .
COPY scaler.pkl .
COPY feature_names.pkl .

# Create non-root user for security
RUN useradd --create-home --shell /bin/bash app_user
RUN chown -R app_user:app_user /app
USER app_user

# Expose port
EXPOSE 5000

# Health check
HEALTHCHECK --interval=30s --timeout=30s --start-period=5s --retries=3 \
    CMD curl -f http://localhost:5000/health || exit 1

# Run application
CMD ["python", "app.py"]
'''

# Save Dockerfile
with open('Dockerfile', 'w') as f:
    f.write(dockerfile_content.strip())

print("✅ Dockerfile created!")

In [ ]:
# Create optimized requirements.txt for Docker
requirements_content = '''
Flask==2.3.3
scikit-learn==1.3.0
pandas==2.0.3
numpy==1.24.3
joblib==1.3.2
gunicorn==21.2.0
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("✅ Requirements.txt created!")

In [ ]:
# Create .dockerignore file
dockerignore_content = '''
# Python
__pycache__/
*.pyc
*.pyo
*.pyd
.Python
env/
venv/
.venv/
pip-log.txt
pip-delete-this-directory.txt
.tox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.log
.git
.mypy_cache
.pytest_cache
.hypothesis

# Jupyter Notebook
.ipynb_checkpoints

# IDE
.vscode/
.idea/
*.swp
*.swo
*~

# OS
.DS_Store
.DS_Store?
._*
.Spotlight-V100
.Trashes
ehthumbs.db
Thumbs.db

# Docker
Dockerfile*
docker-compose*
.dockerignore

# Documentation
README.md
*.md
docs/

# Tests
tests/
test_*.py
*_test.py
'''

with open('.dockerignore', 'w') as f:
    f.write(dockerignore_content.strip())

print("✅ .dockerignore created!")

## 🚀 Production-Ready Flask App

In [ ]:
# Create production-ready Flask app with Gunicorn
production_app_content = '''
import os
import numpy as np
import pandas as pd
import joblib
import json
from flask import Flask, request, jsonify
from datetime import datetime
import logging
import sys

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Initialize Flask app
app = Flask(__name__)

# Configuration
app.config['JSON_SORT_KEYS'] = False
app.config['JSONIFY_PRETTYPRINT_REGULAR'] = True

# Global variables for model artifacts
model = None
scaler = None
feature_names = None
prediction_count = 0

def load_model_artifacts():
    """Load model artifacts with error handling"""
    global model, scaler, feature_names
    
    try:
        model = joblib.load('model.pkl')
        scaler = joblib.load('scaler.pkl')
        feature_names = joblib.load('feature_names.pkl')
        logger.info("Model artifacts loaded successfully")
        return True
    except Exception as e:
        logger.error(f"Error loading model artifacts: {e}")
        return False

def validate_input(data):
    """Validate input data"""
    if not isinstance(data, dict):
        return False, "Input must be a JSON object"
    
    if 'features' not in data:
        return False, "Missing 'features' key in input"
    
    features = data['features']
    
    if not isinstance(features, dict):
        return False, "Features must be a dictionary"
    
    # Check if all required features are present
    missing_features = set(feature_names) - set(features.keys())
    if missing_features:
        return False, f"Missing features: {list(missing_features)}"
    
    # Check if all values are numeric
    for feature, value in features.items():
        if not isinstance(value, (int, float)):
            return False, f"Feature '{feature}' must be numeric"
    
    return True, "Valid input"

@app.before_first_request
def initialize():
    """Initialize application"""
    logger.info("Initializing ML API...")
    if not load_model_artifacts():
        logger.error("Failed to load model artifacts")

@app.route('/health', methods=['GET'])
def health_check():
    """Health check endpoint"""
    if model and scaler and feature_names:
        return jsonify({
            'status': 'healthy',
            'model_loaded': True,
            'predictions_made': prediction_count,
            'timestamp': datetime.now().isoformat(),
            'version': '1.0.0'
        })
    else:
        return jsonify({
            'status': 'unhealthy',
            'model_loaded': False,
            'error': 'Model not loaded'
        }), 500

@app.route('/predict', methods=['POST'])
def predict():
    """Prediction endpoint"""
    global prediction_count
    
    if not (model and scaler and feature_names):
        return jsonify({'error': 'Model not loaded'}), 500
    
    try:
        data = request.get_json()
        if not data:
            return jsonify({'error': 'No JSON data provided'}), 400
        
        # Validate input
        is_valid, message = validate_input(data)
        if not is_valid:
            return jsonify({'error': message}), 400
        
        # Preprocess input
        features = data['features']
        feature_values = [features[name] for name in feature_names]
        X = np.array(feature_values).reshape(1, -1)
        X_scaled = scaler.transform(X)
        
        # Make prediction
        prediction = model.predict(X_scaled)[0]
        probability = model.predict_proba(X_scaled)[0]
        
        # Update prediction count
        prediction_count += 1
        
        # Prepare response
        response = {
            'prediction': int(prediction),
            'probability': {
                'class_0': float(probability[0]),
                'class_1': float(probability[1])
            },
            'confidence': float(max(probability)),
            'timestamp': datetime.now().isoformat(),
            'model_version': '1.0.0'
        }
        
        logger.info(f"Prediction made: {prediction} (confidence: {max(probability):.3f})")
        
        return jsonify(response)
        
    except Exception as e:
        logger.error(f"Prediction error: {e}")
        return jsonify({'error': f'Prediction failed: {str(e)}'}), 500

@app.route('/model-info', methods=['GET'])
def model_info():
    """Model information endpoint"""
    if not (model and scaler and feature_names):
        return jsonify({'error': 'Model not loaded'}), 500
    
    return jsonify({
        'model_type': 'RandomForestClassifier',
        'features': feature_names,
        'num_features': len(feature_names),
        'model_version': '1.0.0',
        'predictions_made': prediction_count
    })

@app.errorhandler(404)
def not_found(error):
    return jsonify({'error': 'Endpoint not found'}), 404

@app.errorhandler(500)
def internal_error(error):
    return jsonify({'error': 'Internal server error'}), 500

if __name__ == '__main__':
    # Load model artifacts on startup
    if load_model_artifacts():
        logger.info("Starting Flask development server...")
        app.run(host='0.0.0.0', port=5000, debug=False)
    else:
        logger.error("Cannot start server: Model artifacts not loaded")
        sys.exit(1)
'''

# Save production app
with open('app_production.py', 'w') as f:
    f.write(production_app_content.strip())

print("✅ Production Flask app created!")

## 🔧 Docker Build and Run Scripts

In [ ]:
# Create Docker build script
build_script = '''
#!/bin/bash

# Docker build script for ML API

set -e  # Exit on any error

# Configuration
IMAGE_NAME="ml-api"
IMAGE_TAG="latest"
FULL_IMAGE_NAME="${IMAGE_NAME}:${IMAGE_TAG}"

echo "🐳 Building Docker image: ${FULL_IMAGE_NAME}"

# Build the image
docker build -t ${FULL_IMAGE_NAME} .

echo "✅ Docker image built successfully!"
echo "📊 Image size:"
docker images ${IMAGE_NAME} --format "table {{.Repository}}\\t{{.Tag}}\\t{{.Size}}"

echo "\n🚀 To run the container:"
echo "   docker run -p 5000:5000 ${FULL_IMAGE_NAME}"
'''

with open('build.sh', 'w') as f:
    f.write(build_script.strip())

# Make script executable
os.chmod('build.sh', 0o755)

print("✅ Docker build script created!")

In [ ]:
# Create Docker run script
run_script = '''
#!/bin/bash

# Docker run script for ML API

set -e  # Exit on any error

# Configuration
IMAGE_NAME="ml-api:latest"
CONTAINER_NAME="ml-api-container"
HOST_PORT="5000"
CONTAINER_PORT="5000"

echo "🐳 Running Docker container: ${CONTAINER_NAME}"

# Stop and remove existing container if it exists
if docker ps -a --format '{{.Names}}' | grep -q "^${CONTAINER_NAME}$"; then
    echo "🛑 Stopping existing container..."
    docker stop ${CONTAINER_NAME}
    docker rm ${CONTAINER_NAME}
fi

# Run the container
docker run -d \
    --name ${CONTAINER_NAME} \
    -p ${HOST_PORT}:${CONTAINER_PORT} \
    --restart unless-stopped \
    ${IMAGE_NAME}

echo "✅ Container started successfully!"
echo "🌐 API available at: http://localhost:${HOST_PORT}"
echo "📊 Container status:"
docker ps --filter "name=${CONTAINER_NAME}" --format "table {{.Names}}\\t{{.Status}}\\t{{.Ports}}"

echo "\n📝 Useful commands:"
echo "   View logs: docker logs ${CONTAINER_NAME}"
echo "   Stop container: docker stop ${CONTAINER_NAME}"
echo "   Remove container: docker rm ${CONTAINER_NAME}"
'''

with open('run.sh', 'w') as f:
    f.write(run_script.strip())

# Make script executable
os.chmod('run.sh', 0o755)

print("✅ Docker run script created!")

## 🎯 Practice Problems

### **Problem 1: Multi-stage Docker Build**
Create a multi-stage Dockerfile to optimize image size.

In [ ]:
def create_multistage_dockerfile():
    """
    Create multi-stage Dockerfile for smaller production images
    
    Stages:
    1. Builder stage - Install dependencies and build
    2. Production stage - Copy only necessary files
    
    Returns:
    str: Multi-stage Dockerfile content
    """
    # Your code here
    pass

# Test your implementation
# multistage_dockerfile = create_multistage_dockerfile()
# print(multistage_dockerfile)

### **Problem 2: Docker Compose for Development**
Create a docker-compose.yml for development environment.

In [ ]:
def create_docker_compose():
    """
    Create docker-compose.yml for development
    
    Include:
    - ML API service
    - Redis for caching
    - PostgreSQL for logging
    - Volume mounts for development
    
    Returns:
    str: Docker Compose YAML content
    """
    # Your code here
    pass

# Test your implementation
# compose_content = create_docker_compose()
# with open('docker-compose.yml', 'w') as f:
#     f.write(compose_content)

## 🎯 Key Takeaways

1. **Docker** ensures consistent deployment across environments
2. **Multi-stage builds** optimize image size for production
3. **Health checks** enable container orchestration
4. **Non-root users** improve container security
5. **Proper logging** aids in debugging and monitoring

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Build and test the Docker image**
3. **Move to the next notebook**: Cloud Deployment

---

**Excellent containerization skills!** 🎉 You can now containerize ML applications for consistent deployment.